In [1]:
import transformers
import accelerate
import bitsandbytes

print(transformers.__version__)
print(accelerate.__version__)
print(bitsandbytes.__version__)

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.9.0
1.13.0
0.49.2


In [2]:
import pandas as pd

from config import *
from utils import load_model
from dataset_loader import build_xquad_dataset
import torch
import gc
from methods import (
    DirectMultilingualRAG,
    TranslateThenGenerate,
    RLMInspiredXLQA,
    RLMInspiredXLQAnew
)

from evaluator import evaluate_output


def main():

    tokenizer, model = load_model()

    dataset = build_xquad_dataset()
    print("Dataset size = ", len(dataset))

    methods = {
        "DirectMultilingualRAG":
            DirectMultilingualRAG(model, tokenizer),

        "TranslateThenGenerate":
            TranslateThenGenerate(model, tokenizer),

        "RLMInspiredXLQA":
            RLMInspiredXLQAnew(model, tokenizer)
    }

    all_results = []

    for idx, sample in enumerate(dataset):

        question = sample["question"]
        reference = sample["reference_answer"]

        en_context = sample["en_context"]
        zh_context = sample["zh_context"]

        for method_name, method in methods.items():

            print(f"Running: {idx} | {method_name}")

            prediction = method.answer(
                question,
                en_context,
                zh_context,
                TARGET_LANGUAGE
            )

            metrics = evaluate_output(
                prediction,
                reference
            )

            row = {
                "sample_id": idx,
                "method": method_name,
                "question": question,
                "prediction": prediction,
                "reference": reference
            }

            row.update(metrics)

            all_results.append(row)
        torch.cuda.empty_cache()
        gc.collect()

    df = pd.DataFrame(all_results)

    df.to_csv(OUTPUT_CSV, index=False)

    print(df.groupby("method").mean(
        numeric_only=True
    ))


if __name__ == "__main__":
    main()

Loading weights: 100%|██████████| 434/434 [00:02<00:00, 147.54it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Dataset size =  200
Running: 0 | DirectMultilingualRAG


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7759.36it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running: 0 | TranslateThenGenerate


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5567.90it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Running: 199 | TranslateThenGenerate


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6923.13it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running: 199 | RLMInspiredXLQA


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7076.56it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                       sample_id  bertscore      bleu       chrf  \
method                                                             
DirectMultilingualRAG       99.5   0.677260  4.598598  27.197162   
RLMInspiredXLQA             99.5   0.710451  9.224948  33.028561   
TranslateThenGenerate       99.5   0.711586  9.205248  34.331515   

                       language_fidelity  code_switch_rate  
method                                                      
DirectMultilingualRAG              1.000          0.256902  
RLMInspiredXLQA                    0.945          0.318291  
TranslateThenGenerate              0.950          0.314240  


In [5]:
print(df.groupby("method").mean(numeric_only=True))

                       sample_id  bertscore      bleu       chrf  \
method                                                             
DirectMultilingualRAG       99.5   0.677260  4.598598  27.197162   
RLMInspiredXLQA             99.5   0.710451  9.224948  33.028561   
TranslateThenGenerate       99.5   0.711586  9.205248  34.331515   

                       language_fidelity  code_switch_rate  
method                                                      
DirectMultilingualRAG              1.000          0.256902  
RLMInspiredXLQA                    0.945          0.318291  
TranslateThenGenerate              0.950          0.314240  


In [3]:
import pandas as pd
df = pd.read_csv("outputs/results.csv")
print(df.head())

   sample_id                 method  \
0          0  DirectMultilingualRAG   
1          0  TranslateThenGenerate   
2          0        RLMInspiredXLQA   
3          1  DirectMultilingualRAG   
4          1  TranslateThenGenerate   

                                            question  \
0           Đội thủ Panthers đã thua bao nhiêu điểm?   
1           Đội thủ Panthers đã thua bao nhiêu điểm?   
2           Đội thủ Panthers đã thua bao nhiêu điểm?   
3  Jared Allen có bao nhiêu lần vật ngã trong sự ...   
4  Jared Allen có bao nhiêu lần vật ngã trong sự ...   

                                          prediction reference  bertscore  \
0         Đội thủ của đội Panthers đã thua 308 điểm.       308   0.614142   
1                 Đội thủ Panthers đã thua 308 điểm.       308   0.625405   
2                    Đội thủ Panthers thua 308 điểm.       308   0.631144   
3  Trong sự nghiệp của Jared Allen, anh đã có 136...       136   0.563156   
4       Jared Allen vật ngã 136 lần trong s

In [4]:
# Load the results
df = pd.read_csv("outputs/results.csv")

# Function to display results for a specific sample
def display_sample_results(df, sample_id):
    """Display all methods' results for a given sample ID"""
    sample_data = df[df['sample_id'] == sample_id]
    
    if len(sample_data) == 0:
        print(f"No data found for sample {sample_id}")
        return
    
    # Get the question (same for all methods)
    question = sample_data.iloc[0]['question']
    reference = sample_data.iloc[0]['reference']
    
    print("=" * 80)
    print(f"SAMPLE {sample_id}")
    print("=" * 80)
    print(f"\nQuestion: {question}")
    print(f"Reference Answer: {reference}")
    print("\n" + "-" * 80)
    
    # Display each method's results
    for idx, row in sample_data.iterrows():
        method = row['method']
        prediction = row['prediction']
        
        print(f"\n【{method}】")
        print(f"Prediction: {prediction}")
        print(f"\nMetrics:")
        print(f"  • BERTScore:         {row['bertscore']:.4f}")
        print(f"  • BLEU:              {row['bleu']:.4f}")
        print(f"  • chrF:              {row['chrf']:.4f}")
        print(f"  • Language Fidelity: {row['language_fidelity']:.2f}")
        print(f"  • Code-Switch Rate:  {row['code_switch_rate']:.4f}")
        print("-" * 80)
    
    print("\n")


# Function to display all samples
def display_all_samples(df):
    """Display results for all samples"""
    unique_samples = sorted(df['sample_id'].unique())
    
    for sample_id in unique_samples:
        display_sample_results(df, sample_id)


# Function to create a comparison table for a specific sample
def create_comparison_table(df, sample_id):
    """Create a side-by-side comparison table for one sample"""
    sample_data = df[df['sample_id'] == sample_id].copy()
    
    if len(sample_data) == 0:
        print(f"No data found for sample {sample_id}")
        return None
    
    # Pivot to create comparison
    comparison = sample_data.set_index('method')[['prediction', 'bertscore', 'bleu', 'chrf', 
                                                    'language_fidelity', 'code_switch_rate']]
    
    print(f"\n{'=' * 80}")
    print(f"COMPARISON TABLE - Sample {sample_id}")
    print(f"Question: {sample_data.iloc[0]['question']}")
    print(f"Reference: {sample_data.iloc[0]['reference']}")
    print(f"{'=' * 80}\n")
    
    return comparison


# Function to highlight the best metric for each sample
def display_sample_with_winners(df, sample_id):
    """Display results with highlighted best scores"""
    sample_data = df[df['sample_id'] == sample_id]
    
    if len(sample_data) == 0:
        print(f"No data found for sample {sample_id}")
        return
    
    question = sample_data.iloc[0]['question']
    reference = sample_data.iloc[0]['reference']
    
    print("=" * 80)
    print(f"SAMPLE {sample_id} - WITH WINNERS")
    print("=" * 80)
    print(f"\nQuestion: {question}")
    print(f"Reference: {reference}")
    print("\n" + "-" * 80)
    
    # Find best scores
    best_bertscore = sample_data['bertscore'].max()
    best_bleu = sample_data['bleu'].max()
    best_chrf = sample_data['chrf'].max()
    best_lang_fid = sample_data['language_fidelity'].max()
    best_code_switch = sample_data['code_switch_rate'].min()  # Lower is better
    
    for idx, row in sample_data.iterrows():
        method = row['method']
        prediction = row['prediction']
        
        print(f"\n【{method}】")
        print(f"Prediction: {prediction}")
        print(f"\nMetrics:")
        
        # Add winner indicators
        bert_mark = " ★ BEST" if row['bertscore'] == best_bertscore else ""
        bleu_mark = " ★ BEST" if row['bleu'] == best_bleu else ""
        chrf_mark = " ★ BEST" if row['chrf'] == best_chrf else ""
        lang_mark = " ★ BEST" if row['language_fidelity'] == best_lang_fid else ""
        code_mark = " ★ BEST" if row['code_switch_rate'] == best_code_switch else ""
        
        print(f"  • BERTScore:         {row['bertscore']:.4f}{bert_mark}")
        print(f"  • BLEU:              {row['bleu']:.4f}{bleu_mark}")
        print(f"  • chrF:              {row['chrf']:.4f}{chrf_mark}")
        print(f"  • Language Fidelity: {row['language_fidelity']:.2f}{lang_mark}")
        print(f"  • Code-Switch Rate:  {row['code_switch_rate']:.4f}{code_mark}")
        print("-" * 80)
    
    print("\n")


# Usage examples:

# 1. Display specific sample (e.g., sample 0)
display_sample_results(df, 0)

# 2. Display specific sample with winners highlighted
display_sample_with_winners(df, 0)

# 3. Display first 5 samples
for i in range(min(5, df['sample_id'].max() + 1)):
    display_sample_with_winners(df, i)

# 4. Create comparison table for a sample
comparison_df = create_comparison_table(df, 0)
if comparison_df is not None:
    print(comparison_df)

# 5. Display all samples (use with caution if you have many samples)
# display_all_samples(df)

SAMPLE 0

Question: Đội thủ Panthers đã thua bao nhiêu điểm?
Reference Answer: 308

--------------------------------------------------------------------------------

【DirectMultilingualRAG】
Prediction: Đội thủ của đội Panthers đã thua 308 điểm.

Metrics:
  • BERTScore:         0.6141
  • BLEU:              4.1961
  • chrF:              24.2034
  • Language Fidelity: 1.00
  • Code-Switch Rate:  0.2222
--------------------------------------------------------------------------------

【TranslateThenGenerate】
Prediction: Đội thủ Panthers đã thua 308 điểm.

Metrics:
  • BERTScore:         0.6254
  • BLEU:              5.5224
  • chrF:              28.3185
  • Language Fidelity: 1.00
  • Code-Switch Rate:  0.2857
--------------------------------------------------------------------------------

【RLMInspiredXLQA】
Prediction: Đội thủ Panthers thua 308 điểm.

Metrics:
  • BERTScore:         0.6311
  • BLEU:              6.5673
  • chrF:              30.0201
  • Language Fidelity: 1.00
  • Code-Sw